# 01 · Exploración del dataset (REES46)
Revisa el stream generado por `src/preparar_datos.py` y obtiene los datos que definen decisiones de diseño:
densidad del grafo de co-compra y umbrales del Count-Min Sketch.

In [ ]:
import json, sys
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '..')
RUTA = '../data/stream_10M.parquet'   # o el .pkl si no se instaló pyarrow
df = pd.read_parquet(RUTA) if RUTA.endswith('.parquet') else pd.read_pickle(RUTA)
df.shape, df.dtypes

In [ ]:
with open('../data/eda_resumen.json', encoding='utf-8') as f:
    resumen = json.load(f)
resumen

## Distribución de eventos y actividad por hora

In [ ]:
real = df[~df['synthetic']]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
real['event_type'].astype(str).value_counts().plot.bar(ax=ax[0], title='Eventos por tipo')
real.set_index('event_time').resample('1h').size().plot(ax=ax[1], title='Eventos por hora')
plt.tight_layout()

## Tamaño de las órdenes (define si el grafo de co-compra es suficientemente denso)

In [ ]:
print('Eventos sintéticos:', resumen['eventos_sinteticos'])
print('Pagos rechazados:', resumen['pagos_rechazados'], '| productos sin stock inicial:', resumen['productos_sin_stock_inicial'])

In [ ]:
compras = real[real['event_type'] == 'purchase']
tam = compras.groupby('user_session')['product_id'].nunique()
print(tam.describe()); print('órdenes con >=2 productos:', (tam >= 2).sum())
vc = tam.clip(upper=10).value_counts().sort_index()
plt.figure(figsize=(6, 3)); plt.bar(vc.index.astype(str), vc.values); plt.title('Productos distintos por orden (10 = 10 o más)')

## Intensidad por usuario en ventanas de 10 minutos (umbral del Count-Min Sketch)

In [ ]:
for tipo in ('cart', 'purchase'):
    x = real[real['event_type'] == tipo].groupby(['user_id', pd.Grouper(key='event_time', freq='10min')]).size()
    print(tipo, x.quantile([0.5, 0.99, 0.999]).to_dict(), 'máx', x.max())

## Precio y popularidad (atributos del KD-Tree)

In [ ]:
prod = real.groupby('product_id').agg(precio=('price', 'median'), eventos=('event_type', 'size'))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
prod['precio'].plot.hist(bins=60, log=True, ax=ax[0], title='Precio por producto')
prod['eventos'].sort_values(ascending=False).reset_index(drop=True).plot(loglog=True, ax=ax[1], title='Popularidad (rango vs eventos)')
plt.tight_layout()